# 1. 학습전 데이터 분포 확인

In [ ]:
import pandas as pd
import json
import numpy as np
from pathlib import Path
import shutil
import os
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

In [ ]:
load_dir = "processed_data/psf_tables/number"

psf_tables = {}

for file in os.listdir(load_dir):

    if file.endswith(".csv"):

        psf_name = file.replace(".csv", "")
        if psf_name == "환경":
            continue  # Skip the "환경" file
        if psf_name.startswith("환경"):
            psf_name = "환경" # Rename files starting with "환경_new_category" to "환경"
        psf_tables[psf_name] = pd.read_csv(
            os.path.join(load_dir, file)
        )

print(psf_tables.keys())

job_latest_file = max(
    Path("processed_data").glob("job_*.csv"),
    key=lambda x: x.stat().st_mtime
)

job_df = pd.read_csv(job_latest_file)

health_latest_file = max(
    Path("processed_data").glob("health_*.csv"),
    key=lambda x: x.stat().st_mtime
)

health_df = pd.read_csv(health_latest_file)

# answer, job, health 데이터 불러오기
answer_latest_file = max(
    Path("processed_data").glob("answer_*.csv"),
    key=lambda x: x.stat().st_mtime
)

answer_df = pd.read_csv(answer_latest_file)

In [ ]:
# 데이터 결측치 확인
for psf, df in psf_tables.items():
    print(psf)
    print(df.isna().sum()[df.isna().sum() > 0])

## 1.1 불성실 응답 비율에 따른 케이스 분리

In [ ]:
for psf_name, df in psf_tables.items():
    print(f"===== {psf_name} =====")
    display(df.head(1))

In [ ]:
# 성실도 기준별 PSF 테이블 생성
quality_psf_tables = {
    "100": {},
    "95": {},
    "90": {}
}

for psf_name, df in psf_tables.items():

    quality_psf_tables["100"][psf_name] = (
        df[df["quality_100"] == "성실"].copy()
    )

    quality_psf_tables["95"][psf_name] = (
        df[df["quality_95"] == "성실"].copy()
    )

    quality_psf_tables["90"][psf_name] = (
        df[df["quality_90"] == "성실"].copy()
    )

## 1.2 응답 데이터 확인
- 불성실 응답률별 관측된 시나리오에서 응답 일치율이 60%이하인 경우의 수와 비율 확인
- 60%이하 응답 비율을 구할 때 단일 응담으로 인한 응답률 오류를 막기 위해 2개 이상 응답에 대해 계산한 컬럼을 따로 구하여 적용

In [ ]:
answer_order = ["매우 높음", "좋음", "보통", "약간 높음", "나쁨", "매우나쁨"]

for psf_name in psf_tables.keys():

    rows = []

    for quality in ["100", "95", "90"]:

        df = quality_psf_tables[quality][psf_name]

        counts = (
            df["answer"]
            .value_counts()
            .reindex(answer_order, fill_value=0)
        )

        row = {
            "quality": quality,
            "전체": len(df),
            **counts.to_dict()
        }

        rows.append(row)

    summary = (
        pd.DataFrame(rows)
        .set_index("quality")
    )

    # print(f"\n===== {psf_name} =====")
    # display(summary)

In [ ]:
# 사용하지 않을 컬럼 목록
base_drop_cols = [
    "empid",
    "survey_id",
    "survey_year",
    "set_id",
    "repeat_no",
    "PSF_name",
    "quality_100",
    "quality_95",
    "quality_90"
]

In [ ]:
LOW_CONSENSUS_RATIO = 60
MIN_CASE_N = 2   # 1이면 모든 단일 케이스가 100%라 의미가 약함

low_consensus_dict = {}

for psf_name in psf_tables.keys():

    low_consensus_dict[psf_name] = {}

    # print("=" * 100)
    # print(f"PSF: {psf_name}")

    for quality in ["100", "95", "90"]:

        df = quality_psf_tables[quality][psf_name].copy()

        group_cols = [
            c for c in df.columns
            if c not in base_drop_cols + ["answer"]
        ]

        case_count = (
            df
            .groupby(group_cols, dropna=False)["answer"]
            .value_counts()
            .unstack(fill_value=0)
            .reindex(columns=answer_order, fill_value=0)
        )

        case_ratio = (
            case_count
            .div(case_count.sum(axis=1), axis=0)
            .mul(100)
            .round(2)
        )

        case_summary = case_count.copy()

        for col in answer_order:
            case_summary[f"{col}_비율"] = case_ratio[col]

        case_summary["n_case"] = case_count.sum(axis=1)
        case_summary["max_ratio"] = case_ratio.max(axis=1)
        case_summary["major_answer"] = case_ratio.idxmax(axis=1)
        case_summary["n_unique_answer"] = (case_count > 0).sum(axis=1)

        low_consensus = (
            case_summary[
                (case_summary["max_ratio"] <= LOW_CONSENSUS_RATIO)
                & (case_summary["n_case"] >= MIN_CASE_N)
            ]
            .sort_values(
                by=["max_ratio", "n_case"],
                ascending=[True, False]
            )
            .reset_index()
        )

        low_consensus_dict[psf_name][quality] = low_consensus

        # print(f"\nQuality {quality}")
        # print("row 수:", len(df))
        # print("feature 수:", len(group_cols))
        # print("전체 시나리오 수:", len(case_summary))
        # print("과반 비율 60% 이하 시나리오 수:", len(low_consensus))
        # print(
        #     "비율:",
        #     round(len(low_consensus) / len(case_summary) * 100, 2)
        #     if len(case_summary) > 0 else 0
        # )

        # display(low_consensus)

In [ ]:
summary_rows = []

for psf_name in psf_tables.keys():

    # 원본 테이블 기준(가능한 시나리오 타입 계산용)
    base_df = psf_tables[psf_name]

    group_cols = [
        c for c in base_df.columns
        if c not in base_drop_cols + ["answer"]
    ]

    # 가능한 시나리오 타입 수
    possible_case = 1
    for col in group_cols:
        possible_case *= base_df[col].nunique(dropna=False)

    row = {
        "PSF": psf_name,
        "가능시나리오": possible_case
    }

    for quality in ["100", "95", "90"]:

        df = quality_psf_tables[quality][psf_name]

        observed_case = df.groupby(group_cols, dropna=False).ngroups
        low_case = len(low_consensus_dict[psf_name][quality])

        row[f"{quality}_관측"] = observed_case
        row[f"{quality}_관측률"] = f"{observed_case / possible_case * 100:.2f}%"

        row[f"{quality}_60%↓"] = low_case
        row[f"{quality}_60%↓비율"] = (
            f"{low_case / observed_case * 100:.2f}%"
            if observed_case > 0 else "-"
        )

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

display(summary_df)

In [ ]:
summary_rows = []

for psf_name in psf_tables.keys():

    base_df = psf_tables[psf_name]

    group_cols = [
        c for c in base_df.columns
        if c not in base_drop_cols + ["answer"]
    ]

    # 가능한 시나리오 수
    possible_case = 1
    for col in group_cols:
        possible_case *= base_df[col].nunique(dropna=False)

    row = {
        "PSF": psf_name,
        "가능시나리오": possible_case
    }

    # 95%만 출력
    quality = "95"

    df = quality_psf_tables[quality][psf_name]

    observed_case = df.groupby(group_cols, dropna=False).ngroups

    # 시나리오별 응답수
    scenario_size = (
        df.groupby(group_cols, dropna=False)
          .size()
    )

    # 응답이 2명 이상인 시나리오
    valid_scenarios = scenario_size[scenario_size >= 2]
    valid_case = len(valid_scenarios)

    # 기존 low consensus
    low_case = len(low_consensus_dict[psf_name][quality])

    # low consensus 중 응답이 2명 이상인 것만
    low_valid_case = (
        low_consensus_dict[psf_name][quality]
        .merge(
            valid_scenarios.rename("n"),
            left_on=group_cols,
            right_index=True,
            how="inner"
        )
        .shape[0]
    )

    row["95_60%↓"] = low_case
    row["95_60%↓비율"] = (
        f"{low_case / observed_case * 100:.2f}%"
        if observed_case > 0 else "-"
    )

    row["응답2명이상"] = valid_case

    row["응답2명이상_60%↓"] = low_valid_case
    row["응답2명이상_60%↓비율"] = (
        f"{low_valid_case / valid_case * 100:.2f}%"
        if valid_case > 0 else "-"
    )

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

display(summary_df)

In [ ]:
quality = "95"

summary_rows = []

for psf_name in psf_tables.keys():

    df = quality_psf_tables[quality][psf_name].copy()

    group_cols = [
        c for c in df.columns
        if c not in base_drop_cols + ["answer"]
    ]

    scenario_size = (
        df.groupby(group_cols, dropna=False)
          .size()
          .rename("n_response")
          .reset_index()
    )

    total_response = len(df)
    total_scenario = len(scenario_size)

    # 응답 2명 이상 시나리오의 응답 수
    valid_response_count = (
        scenario_size.loc[
            scenario_size["n_response"] >= 2,
            "n_response"
        ].sum()
    )

    # 응답 2명 이상 시나리오 수
    valid_scenario_count = (
        scenario_size["n_response"] >= 2
    ).sum()

    # 60% 이하 시나리오
    low_df = low_consensus_dict[psf_name][quality]

    low_response = low_df.merge(
        scenario_size,
        on=group_cols,
        how="left"
    )

    low_response_count = low_response["n_response"].sum()

    row = {
        "PSF": psf_name,

        "전체시나리오수": total_scenario,
        "전체응답수": total_response,

        "응답2명이상_시나리오수": valid_scenario_count,
        "응답2명이상_응답수": valid_response_count,

        "60%↓_응답수": low_response_count,

        "60%↓_전체응답비율": (
            f"{low_response_count / total_response * 100:.2f}%"
            if total_response > 0 else "-"
        ),

        "60%↓_응답2명이상기준비율": (
            f"{low_response_count / valid_response_count * 100:.2f}%"
            if valid_response_count > 0 else "-"
        )
    }

    summary_rows.append(row)

summary_response_df = pd.DataFrame(summary_rows)

display(summary_response_df)

# 2. Catboost 모델 실행
- 95%의 성실도에 대해 psf별로 모델을 돌려 검증
- 시나리오별 응답 일치율이 60%이하인 경우가 많기에 이론적 최대 정확도 개념 적용
- 시나리오별 응답 수가 적은 경우도 있어 GroupKFold 적용하여 학습, 테스트 케이스 생성
- fold는 5번 반복하여 수행

In [ ]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
load_dir = "processed_data/psf_tables/number"

psf_tables = {}

processed = {"환경", "복잡성", "교육_및_훈련", "장비_및_기기"}

for file in os.listdir(load_dir):

    if file.endswith(".csv"):

        psf_name = file.replace(".csv", "")
        if psf_name in processed:
            continue  # Skip files not in the processed set
        if psf_name.endswith("_new_category"):
            psf_name = psf_name.replace("_new_category", "")  # Remove the suffix for consistency
        
        psf_tables[psf_name] = pd.read_csv(
            os.path.join(load_dir, file)
        )

In [ ]:
# 성실도 기준별 PSF 테이블 생성
quality_psf_tables = {
    "100": {},
    "95": {},
    "90": {}
}

for psf_name, df in psf_tables.items():

    quality_psf_tables["100"][psf_name] = (
        df[df["quality_100"] == "성실"].copy()
    )

    quality_psf_tables["95"][psf_name] = (
        df[df["quality_95"] == "성실"].copy()
    )

    quality_psf_tables["90"][psf_name] = (
        df[df["quality_90"] == "성실"].copy()
    )

In [ ]:
quality_psf_tables["100"]["복잡성"].head(1)

In [ ]:
quality = "95"
target_col = "answer"

drop_base_cols = [
    "empid",
    "survey_id",
    "survey_year",
    "set_id",
    "repeat_no",
    "PSF_name",
    "quality_100",
    "quality_95",
    "quality_90",
    target_col,
]

In [ ]:
# 동일한 조건 조합을 하나의 scenario_id로 묶는 함수
def make_scenario_id(df, feature_cols):

    scenario_key = (
        df[feature_cols]
        .astype(str)
        .agg("|".join, axis=1)
    )

    return pd.factorize(scenario_key)[0]
# 모델 예측 일치도 / 해당 시나리오에서 가능한 최대 일치도 계산 함수

def theoretical_max_accuracy(y_true, y_pred, scenario_id):

    eval_df = pd.DataFrame({
        "scenario_id": scenario_id,
        "y_true": y_true,
        "y_pred": y_pred
    })

    numerator = 0
    denominator = 0

    for _, g in eval_df.groupby("scenario_id"):

        true_counts = g["y_true"].value_counts()

        # 모델 예측이 맞춘 개수
        matched = (g["y_true"] == g["y_pred"]).sum()

        # 해당 테스트 시나리오에서 가능한 최대 정답 수
        max_possible = true_counts.max()

        numerator += matched
        denominator += max_possible

    if denominator == 0:
        return np.nan, numerator, denominator

    return numerator / denominator, numerator, denominator

In [ ]:
def run_catboost_by_psf(
    psf_name,
    df,
    n_splits=5,
    random_seed=42,
    verbose=False
):
    df = df.copy()

    # 사용할 feature 컬럼
    feature_cols = [
        c for c in df.columns
        if c not in drop_base_cols
    ]

    # 결측 target 제거
    df = df[df[target_col].notna()].copy()

    # scenario_id 생성
    df["scenario_id"] = make_scenario_id(df, feature_cols)

    X = df[feature_cols]
    y = df[target_col]
    groups = df["scenario_id"]

    # 범주형 feature 지정
    cat_features = list(X.columns)

    n_unique_scenarios = groups.nunique()

    if n_unique_scenarios < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "시나리오 수가 2개 미만",
        }

    n_splits = min(n_splits, n_unique_scenarios)

    gkf = GroupKFold(n_splits=n_splits)

    fold_results = []
    all_y_true = []
    all_y_pred = []
    all_scenario_id = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        scenario_test = groups.iloc[test_idx]

        model = CatBoostClassifier(
            iterations=500,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=20,
            grow_policy="Lossguide",
            max_leaves=32,

            min_data_in_leaf=2,
            random_strength=10,
            bagging_temperature=2,
            loss_function="MultiClass",
            eval_metric="TotalF1",
            random_seed=random_seed,
            verbose=False
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        y_pred = model.predict(X_test).ravel()

        raw_acc = accuracy_score(y_test, y_pred)

        theo_acc, numerator, denominator = theoretical_max_accuracy(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        fold_results.append({
            "psf_name": psf_name,
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_train_scenario": groups.iloc[train_idx].nunique(),
            "n_test_scenario": scenario_test.nunique(),
            "raw_accuracy": raw_acc,
            "theoretical_accuracy": theo_acc,
            "numerator": numerator,
            "denominator": denominator
        })

        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        all_scenario_id.extend(scenario_test.values)

    total_theo_acc, total_num, total_den = theoretical_max_accuracy(
        y_true=np.array(all_y_true),
        y_pred=np.array(all_y_pred),
        scenario_id=np.array(all_scenario_id)
    )

    result = {
        "psf_name": psf_name,
        "status": "success",
        "n_data": len(df),
        "n_scenario": df["scenario_id"].nunique(),
        "n_features": len(feature_cols),
        "feature_cols": feature_cols,
        "fold_results": pd.DataFrame(fold_results),
        "raw_accuracy_mean": pd.DataFrame(fold_results)["raw_accuracy"].mean(),
        "theoretical_accuracy_total": total_theo_acc,
        "theoretical_numerator": total_num,
        "theoretical_denominator": total_den,
        "y_true": np.array(all_y_true),
        "y_pred": np.array(all_y_pred),
    }

    return result

In [ ]:
psf_model_results = {}

for psf_name, df in quality_psf_tables["95"].items():

    print(f"\n==============================")
    print(f"[{psf_name}] 모델 학습 시작")
    print(f"==============================")

    result = run_catboost_by_psf(
        psf_name=psf_name,
        df=df,
        n_splits=5,
        random_seed=42
    )

    psf_model_results[psf_name] = result

    if result["status"] == "success":
        print(f"데이터 수: {result['n_data']}")
        print(f"시나리오 수: {result['n_scenario']}")
        print(f"변수 수: {result['n_features']}")
        print(f"일반 Accuracy 평균: {result['raw_accuracy_mean']:.4f}")
        print(f"이론적 최대 정확도 보정 Accuracy: {result['theoretical_accuracy_total']:.4f}")
        print(
            f"분자/분모: "
            f"{result['theoretical_numerator']} / {result['theoretical_denominator']}"
        )
    else:
        print(f"Skip: {result['reason']}")

In [ ]:
summary_rows = []

for psf_name, result in psf_model_results.items():

    if result["status"] != "success":
        continue

    summary_rows.append({
        "PSF": psf_name,
        "n_data": result["n_data"],
        "n_scenario": result["n_scenario"],
        "n_features": result["n_features"],
        "raw_accuracy_mean": result["raw_accuracy_mean"],
        "theoretical_accuracy": result["theoretical_accuracy_total"],
        "theoretical_numerator": result["theoretical_numerator"],
        "theoretical_denominator": result["theoretical_denominator"],
    })

psf_summary = pd.DataFrame(summary_rows)

psf_summary.sort_values(
    "theoretical_accuracy",
    ascending=False
)

## 2.2 모델 저장

In [ ]:
from pathlib import Path
from catboost import CatBoostClassifier

model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

final_psf_models = {}

for psf_name, df in quality_psf_tables["95"].items():

    print(f"[{psf_name}] 최종 모델 학습")

    df = df.copy()
    df = df[df[target_col].notna()].copy()

    feature_cols = [
        c for c in df.columns
        if c not in drop_base_cols
    ]

    X = df[feature_cols]
    y = df[target_col]

    model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=20,
        grow_policy="Lossguide",
        max_leaves=32,
        min_data_in_leaf=2,
        random_strength=10,
        bagging_temperature=2,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        random_seed=42,
        verbose=False
    )

    # 전체 데이터로 딱 한 번 학습
    model.fit(
        X,
        y,
        cat_features=list(X.columns)
    )

    # 저장
    save_path = model_dir / f"{psf_name}_catboost.cbm"
    model.save_model(str(save_path))

    final_psf_models[psf_name] = {
        "model": model,
        "feature_cols": feature_cols
    }

    print(f"저장 완료: {save_path}")

In [ ]:
psf_model_results["복잡성"]["fold_results"]

In [ ]:
psf_name = "복잡성"

print(
    classification_report(
        psf_model_results[psf_name]["y_true"],
        psf_model_results[psf_name]["y_pred"]
    )
)

In [ ]:
psf_name = "복잡성"

labels = sorted(pd.unique(psf_model_results[psf_name]["y_true"]))

cm = confusion_matrix(
    psf_model_results[psf_name]["y_true"],
    psf_model_results[psf_name]["y_pred"],
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels]
)

cm_df

# 3. Catboost 모델 실행 (장비 및 기기, 복잡성, 교육 및 훈련)
- 위 세 요소는 정확도가 낮게 관측되었음
- 장비 및, 기기, 교육 및 훈련의 경우 학습 데이터 부족으로 인해 실제 모수를 반영하지 못한 것으로 추정
- 추가 데이터 조달이 필요

## 3.1 최소 응답 수별 예측 수행
- 장비 및 기기, 교육 및 훈련의 경우 60%이하의 일치율을 가진 응답이 약 40%정도를 차지하나 단일 응답의 개수가 많은 상태
- 이에 시나리오별 최소 응답수가 1, 2, 3일 때 수행하여 예측률을 확인

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import shutil

from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
load_dir = "processed_data/psf_tables/number"

psf_tables = {}

for file in os.listdir(load_dir):

    if file.endswith(".csv"):

        psf_name = file.replace(".csv", "")
        if psf_name == "환경":
            continue  # Skip the "환경" file
        if psf_name.startswith("환경"):
            psf_name = "환경" # Rename files starting with "환경_new_category" to "환경"
        psf_tables[psf_name] = pd.read_csv(
            os.path.join(load_dir, file)
        )


In [ ]:
quality_psf_tables = {
    "100": {},
    "95": {},
    "90": {}
}

for psf_name, df in psf_tables.items():

    quality_psf_tables["100"][psf_name] = df[df["quality_100"] == "성실"].copy()
    quality_psf_tables["95"][psf_name] = df[df["quality_95"] == "성실"].copy()
    quality_psf_tables["90"][psf_name] = df[df["quality_90"] == "성실"].copy()

In [ ]:
quality = "95"
target_col = "answer"

drop_base_cols = [
    "empid",
    "survey_id",
    "survey_year",
    "set_id",
    "repeat_no",
    "PSF_name",
    "quality_100",
    "quality_95",
    "quality_90",
    target_col,
]

In [ ]:
def make_scenario_id(df, feature_cols):
    scenario_key = (
        df[feature_cols]
        .astype(str)
        .agg("|".join, axis=1)
    )

    return pd.factorize(scenario_key)[0]
def add_scenario_response_count(df, feature_cols):
    df = df.copy()

    scenario_count = (
        df.groupby(feature_cols)
          .size()
          .reset_index(name="scenario_response_count")
    )

    df = df.merge(
        scenario_count,
        on=feature_cols,
        how="left"
    )

    return df
def theoretical_max_accuracy(y_true, y_pred, scenario_id):

    eval_df = pd.DataFrame({
        "scenario_id": scenario_id,
        "y_true": y_true,
        "y_pred": y_pred
    })

    numerator = 0
    denominator = 0

    for _, g in eval_df.groupby("scenario_id"):

        true_counts = g["y_true"].value_counts()

        matched = (g["y_true"] == g["y_pred"]).sum()
        max_possible = true_counts.max()

        numerator += matched
        denominator += max_possible

    if denominator == 0:
        return np.nan, numerator, denominator

    return numerator / denominator, numerator, denominator

In [ ]:
def run_catboost_by_psf(
    psf_name,
    df,
    min_response_per_scenario=1,
    n_splits=5,
    random_seed=42
):
    df = df.copy()

    # 결측 target 제거
    df = df[df[target_col].notna()].copy()

    # feature 컬럼
    feature_cols = [
        c for c in df.columns
        if c not in drop_base_cols
    ]

    # 시나리오별 응답 수 추가
    df = add_scenario_response_count(df, feature_cols)

    # 응답 수 기준 필터링
    df = df[
        df["scenario_response_count"] >= min_response_per_scenario
    ].copy()

    if len(df) < 30:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": f"데이터 수 부족: {len(df)}",
            "min_response_per_scenario": min_response_per_scenario
        }

    # scenario_response_count는 모델 feature에서 제외
    model_feature_cols = [
        c for c in feature_cols
        if c != "scenario_response_count"
    ]

    # scenario_id 생성
    df["scenario_id"] = make_scenario_id(df, model_feature_cols)

    X = df[model_feature_cols]
    y = df[target_col]
    groups = df["scenario_id"]

    cat_features = list(X.columns)

    n_unique_scenarios = groups.nunique()

    if n_unique_scenarios < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "시나리오 수가 2개 미만",
            "min_response_per_scenario": min_response_per_scenario
        }

    n_splits = min(n_splits, n_unique_scenarios)

    if n_splits < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "GroupKFold 불가능",
            "min_response_per_scenario": min_response_per_scenario
        }

    gkf = GroupKFold(n_splits=n_splits)

    fold_results = []
    all_y_true = []
    all_y_pred = []
    all_scenario_id = []

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups),
        start=1
    ):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        scenario_test = groups.iloc[test_idx]

        # train에 클래스가 2개 미만이면 학습 불가
        if y_train.nunique() < 2:
            continue

        model = CatBoostClassifier(
            iterations=500,
            depth=6,
            learning_rate=0.05,
            loss_function="MultiClass",
            eval_metric="TotalF1",
            random_seed=random_seed,
            verbose=False
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        y_pred = model.predict(X_test).ravel()

        raw_acc = accuracy_score(y_test, y_pred)

        theo_acc, numerator, denominator = theoretical_max_accuracy(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        fold_results.append({
            "psf_name": psf_name,
            "min_response_per_scenario": min_response_per_scenario,
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_train_scenario": groups.iloc[train_idx].nunique(),
            "n_test_scenario": scenario_test.nunique(),
            "raw_accuracy": raw_acc,
            "theoretical_accuracy": theo_acc,
            "numerator": numerator,
            "denominator": denominator
        })

        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        all_scenario_id.extend(scenario_test.values)

    if len(fold_results) == 0:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "유효한 fold 없음",
            "min_response_per_scenario": min_response_per_scenario
        }

    fold_df = pd.DataFrame(fold_results)

    total_theo_acc, total_num, total_den = theoretical_max_accuracy(
        y_true=np.array(all_y_true),
        y_pred=np.array(all_y_pred),
        scenario_id=np.array(all_scenario_id)
    )

    result = {
        "psf_name": psf_name,
        "status": "success",
        "min_response_per_scenario": min_response_per_scenario,
        "n_data": len(df),
        "n_scenario": df["scenario_id"].nunique(),
        "n_features": len(model_feature_cols),
        "feature_cols": model_feature_cols,
        "fold_results": fold_df,
        "raw_accuracy_mean": fold_df["raw_accuracy"].mean(),
        "theoretical_accuracy_total": total_theo_acc,
        "theoretical_numerator": total_num,
        "theoretical_denominator": total_den,
        "y_true": np.array(all_y_true),
        "y_pred": np.array(all_y_pred),
    }

    return result

In [ ]:
target_psfs = [
    "장비_및_기기",
    "교육_및_훈련",
    "복잡성"
]

min_response_list = [1, 2, 3]

psf_model_results_by_min_n = {}

for psf_name in target_psfs:

    if psf_name not in quality_psf_tables[quality]:
        print(f"{psf_name} 없음")
        continue

    df = quality_psf_tables[quality][psf_name]

    psf_model_results_by_min_n[psf_name] = {}

    for min_n in min_response_list:

        print(f"\n==============================")
        print(f"[{psf_name}] min_n = {min_n} 모델 학습 시작")
        print(f"==============================")

        result = run_catboost_by_psf(
            psf_name=psf_name,
            df=df,
            min_response_per_scenario=min_n,
            n_splits=5,
            random_seed=42
        )

        psf_model_results_by_min_n[psf_name][min_n] = result

        if result["status"] == "success":
            print(f"데이터 수: {result['n_data']}")
            print(f"시나리오 수: {result['n_scenario']}")
            print(f"변수 수: {result['n_features']}")
            print(f"일반 Accuracy 평균: {result['raw_accuracy_mean']:.4f}")
            print(f"이론적 최대 정확도 보정 Accuracy: {result['theoretical_accuracy_total']:.4f}")
            print(
                f"분자/분모: "
                f"{result['theoretical_numerator']} / {result['theoretical_denominator']}"
            )
        else:
            print(f"Skip: {result['reason']}")

In [ ]:
summary_rows = []

for psf_name, min_results in psf_model_results_by_min_n.items():

    for min_n, result in min_results.items():

        if result["status"] != "success":
            summary_rows.append({
                "PSF": psf_name,
                "min_response_per_scenario": min_n,
                "status": result["status"],
                "reason": result.get("reason", None),
                "n_data": np.nan,
                "n_scenario": np.nan,
                "raw_accuracy_mean": np.nan,
                "theoretical_accuracy": np.nan,
                "theoretical_numerator": np.nan,
                "theoretical_denominator": np.nan,
            })
            continue

        summary_rows.append({
            "PSF": psf_name,
            "min_response_per_scenario": min_n,
            "status": result["status"],
            "reason": None,
            "n_data": result["n_data"],
            "n_scenario": result["n_scenario"],
            "n_features": result["n_features"],
            "raw_accuracy_mean": result["raw_accuracy_mean"],
            "theoretical_accuracy": result["theoretical_accuracy_total"],
            "theoretical_numerator": result["theoretical_numerator"],
            "theoretical_denominator": result["theoretical_denominator"],
        })

psf_response_count_summary = pd.DataFrame(summary_rows)

display(
    psf_response_count_summary.sort_values(
        ["PSF", "min_response_per_scenario"]
    )
)

In [ ]:
pivot_summary = psf_response_count_summary.pivot_table(
    index="PSF",
    columns="min_response_per_scenario",
    values=[
        "n_data",
        "n_scenario",
        "raw_accuracy_mean",
        "theoretical_accuracy"
    ],
    aggfunc="first"
)

display(pivot_summary)

## 3.2 교육 및 훈련 모델 상세 분석 수행
- depth, l2_leaf_reg, random_strength, iterations를 바꿔보며 예측율 성능 확인

In [ ]:
load_dir = "processed_data/psf_tables/number"

psf_tables = {}

for file in os.listdir(load_dir):

    if file.endswith(".csv"):

        psf_name = file.replace(".csv", "")
        if psf_name == "환경":
            continue  # Skip the "환경" file
        if psf_name.startswith("환경"):
            psf_name = "환경" # Rename files starting with "환경_new_category" to "환경"
        psf_tables[psf_name] = pd.read_csv(
            os.path.join(load_dir, file)
        )


In [ ]:
quality_psf_tables = {
    "100": {},
    "95": {},
    "90": {}
}

for psf_name, df in psf_tables.items():

    quality_psf_tables["100"][psf_name] = df[df["quality_100"] == "성실"].copy()
    quality_psf_tables["95"][psf_name] = df[df["quality_95"] == "성실"].copy()
    quality_psf_tables["90"][psf_name] = df[df["quality_90"] == "성실"].copy()

In [ ]:
target_col = "answer"

drop_base_cols = [
    "empid",
    "survey_id",
    "survey_year",
    "set_id",
    "repeat_no",
    "PSF_name",
    "quality_100",
    "quality_95",
    "quality_90",
    target_col,
]

In [ ]:
def get_feature_cols(df, target_col="answer", drop_base_cols=None):
    if drop_base_cols is None:
        drop_base_cols = []

    return [
        c for c in df.columns
        if c not in drop_base_cols
    ]


def make_scenario_id(df, feature_cols):
    scenario_key = (
        df[feature_cols]
        .astype(str)
        .agg("|".join, axis=1)
    )

    return pd.factorize(scenario_key)[0]


def add_scenario_response_count(df, feature_cols):
    df = df.copy()

    scenario_count = (
        df.groupby(feature_cols)
          .size()
          .reset_index(name="scenario_response_count")
    )

    df = df.merge(
        scenario_count,
        on=feature_cols,
        how="left"
    )

    return df

In [ ]:
def theoretical_max_accuracy(y_true, y_pred, scenario_id):

    eval_df = pd.DataFrame({
        "scenario_id": scenario_id,
        "y_true": y_true,
        "y_pred": y_pred
    })

    numerator = 0
    denominator = 0

    for _, g in eval_df.groupby("scenario_id"):

        true_counts = g["y_true"].value_counts()

        matched = (g["y_true"] == g["y_pred"]).sum()
        max_possible = true_counts.max()

        numerator += matched
        denominator += max_possible

    if denominator == 0:
        return np.nan, numerator, denominator

    return numerator / denominator, numerator, denominator

In [ ]:
def run_catboost_by_psf(
    psf_name,
    df,
    min_response_per_scenario=1,
    model_params=None,
    n_splits=5,
    random_seed=42,
    target_col="answer",
    drop_base_cols=None
):
    df = df.copy()

    if drop_base_cols is None:
        drop_base_cols = []

    if model_params is None:
        model_params = {}

    df = df[df[target_col].notna()].copy()

    feature_cols = get_feature_cols(
        df=df,
        target_col=target_col,
        drop_base_cols=drop_base_cols
    )

    df = add_scenario_response_count(df, feature_cols)

    df = df[
        df["scenario_response_count"] >= min_response_per_scenario
    ].copy()

    if len(df) < 30:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": f"데이터 수 부족: {len(df)}",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    model_feature_cols = feature_cols.copy()

    df["scenario_id"] = make_scenario_id(df, model_feature_cols)

    X = df[model_feature_cols]
    y = df[target_col]
    groups = df["scenario_id"]

    cat_features = list(X.columns)

    n_unique_scenarios = groups.nunique()

    if n_unique_scenarios < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "시나리오 수가 2개 미만",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    n_splits = min(n_splits, n_unique_scenarios)

    if n_splits < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "GroupKFold 불가능",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    gkf = GroupKFold(n_splits=n_splits)

    fold_results = []
    all_y_true = []
    all_y_pred = []
    all_scenario_id = []

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups),
        start=1
    ):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        scenario_test = groups.iloc[test_idx]

        if y_train.nunique() < 2:
            continue

        params = {
            "iterations": 500,
            "depth": 4,
            "learning_rate": 0.05,
            "loss_function": "MultiClass",
            "eval_metric": "TotalF1",
            "random_seed": random_seed,
            "verbose": False
        }

        params.update(model_params)

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        y_pred = model.predict(X_test).ravel()

        raw_acc = accuracy_score(y_test, y_pred)

        theo_acc, numerator, denominator = theoretical_max_accuracy(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        fold_results.append({
            "psf_name": psf_name,
            "min_response_per_scenario": min_response_per_scenario,
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_train_scenario": groups.iloc[train_idx].nunique(),
            "n_test_scenario": scenario_test.nunique(),
            "raw_accuracy": raw_acc,
            "theoretical_accuracy": theo_acc,
            "numerator": numerator,
            "denominator": denominator,
            **model_params
        })

        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        all_scenario_id.extend(scenario_test.values)

    if len(fold_results) == 0:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "유효한 fold 없음",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    fold_df = pd.DataFrame(fold_results)

    total_theo_acc, total_num, total_den = theoretical_max_accuracy(
        y_true=np.array(all_y_true),
        y_pred=np.array(all_y_pred),
        scenario_id=np.array(all_scenario_id)
    )

    return {
        "psf_name": psf_name,
        "status": "success",
        "min_response_per_scenario": min_response_per_scenario,
        "n_data": len(df),
        "n_scenario": df["scenario_id"].nunique(),
        "n_features": len(model_feature_cols),
        "feature_cols": model_feature_cols,
        "fold_results": fold_df,
        "raw_accuracy_mean": fold_df["raw_accuracy"].mean(),
        "theoretical_accuracy_total": total_theo_acc,
        "theoretical_numerator": total_num,
        "theoretical_denominator": total_den,
        "y_true": np.array(all_y_true),
        "y_pred": np.array(all_y_pred),
        **model_params
    }

In [ ]:
from itertools import product

param_grid = []

for depth, iterations, l2_leaf_reg, random_strength in product(
    [4, 5],
    [300],
    [20, 50, 100],
    [0]
):
    param_grid.append({
        "depth": depth,
        "iterations": iterations,
        "l2_leaf_reg": l2_leaf_reg,
        "random_strength": random_strength,
        "learning_rate": 0.05
    })

len(param_grid)

In [ ]:
quality = "95"

target_psfs = [
    "교육_및_훈련", "복잡성", "장비_및_기기"
]

min_response_list = [2]

psf_tuning_results = {}

for psf_name in target_psfs:

    if psf_name not in quality_psf_tables[quality]:
        print(f"{psf_name} 없음")
        continue

    df = quality_psf_tables[quality][psf_name]

    psf_tuning_results[psf_name] = {}

    for min_n in min_response_list:

        psf_tuning_results[psf_name][min_n] = []

        for i, model_params in enumerate(param_grid, start=1):

            print(f"\n[{psf_name}] min_n={min_n} / {i}/{len(param_grid)}")
            print(model_params)

            result = run_catboost_by_psf(
                psf_name=psf_name,
                df=df,
                min_response_per_scenario=min_n,
                model_params=model_params,
                n_splits=5,
                random_seed=42,
                target_col=target_col,
                drop_base_cols=drop_base_cols
            )

            psf_tuning_results[psf_name][min_n].append(result)

            if result["status"] == "success":
                print(
                    f"raw={result['raw_accuracy_mean']:.4f}, "
                    f"theo={result['theoretical_accuracy_total']:.4f}, "
                    f"n={result['n_data']}, "
                    f"scenario={result['n_scenario']}"
                )
            else:
                print(f"Skip: {result['reason']}")

In [ ]:
summary_rows = []

for psf_name, min_results in psf_tuning_results.items():

    for min_n, results in min_results.items():

        for result in results:

            row = {
                "PSF": psf_name,
                "min_response_per_scenario": min_n,
                "status": result["status"],
                "reason": result.get("reason", None),
                "n_data": result.get("n_data", np.nan),
                "n_scenario": result.get("n_scenario", np.nan),
                "n_features": result.get("n_features", np.nan),
                "raw_accuracy_mean": result.get("raw_accuracy_mean", np.nan),
                "theoretical_accuracy": result.get("theoretical_accuracy_total", np.nan),
                "theoretical_numerator": result.get("theoretical_numerator", np.nan),
                "theoretical_denominator": result.get("theoretical_denominator", np.nan),
                "depth": result.get("depth", np.nan),
                "iterations": result.get("iterations", np.nan),
                "learning_rate": result.get("learning_rate", np.nan),
                "l2_leaf_reg": result.get("l2_leaf_reg", np.nan),
                "random_strength": result.get("random_strength", np.nan),
            }

            summary_rows.append(row)

psf_tuning_summary = pd.DataFrame(summary_rows)

display(
    psf_tuning_summary.sort_values(
        ["PSF", "min_response_per_scenario", "theoretical_accuracy"],
        ascending=[True, True, False]
    )
)

## 3.3 3개 PSF 수행 V2

In [ ]:
def get_feature_cols(df, target_col="answer", drop_base_cols=None):
    if drop_base_cols is None:
        drop_base_cols = []

    exclude_cols = set(drop_base_cols + [
        target_col,
        "scenario_response_count",
        "scenario_id"
    ])

    return [
        c for c in df.columns
        if c not in exclude_cols
    ]


def run_catboost_by_psf(
    psf_name,
    df,
    min_response_per_scenario=1,   # 학습에 사용할 최소 시나리오 응답 수
    model_params=None,
    n_splits=5,
    random_seed=42,
    target_col="answer",
    drop_base_cols=None
):
    df = df.copy()

    if drop_base_cols is None:
        drop_base_cols = []

    if model_params is None:
        model_params = {}

    df = df[df[target_col].notna()].copy()

    feature_cols = get_feature_cols(
        df=df,
        target_col=target_col,
        drop_base_cols=drop_base_cols
    )

    # 전체 데이터 기준 시나리오 응답 수 계산
    df = add_scenario_response_count(df, feature_cols)

    # 전체 데이터 기준 scenario_id 생성
    df["scenario_id"] = make_scenario_id(df, feature_cols)

    if len(df) < 30:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": f"평가 데이터 수 부족: {len(df)}",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    X = df[feature_cols]
    y = df[target_col]
    groups = df["scenario_id"]

    cat_features = list(X.columns)

    n_unique_scenarios = groups.nunique()
    n_splits = min(n_splits, n_unique_scenarios)

    if n_splits < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "GroupKFold 불가능",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    gkf = GroupKFold(n_splits=n_splits)

    fold_results = []
    all_y_true = []
    all_y_pred = []
    all_scenario_id = []

    for fold, (train_idx_all, test_idx) in enumerate(
        gkf.split(X, y, groups),
        start=1
    ):
        train_df_all = df.iloc[train_idx_all].copy()
        test_df = df.iloc[test_idx].copy()

        # 핵심 수정:
        # 학습 데이터만 min_response_per_scenario 조건 적용
        train_df = train_df_all[
            train_df_all["scenario_response_count"] >= min_response_per_scenario
        ].copy()

        if len(train_df) == 0:
            continue

        if train_df[target_col].nunique() < 2:
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[target_col]

        # 테스트는 단일응답 포함 전체 사용
        X_test = test_df[feature_cols]
        y_test = test_df[target_col]
        scenario_test = test_df["scenario_id"]

        params = {
            "iterations": 500,
            "depth": 4,
            "learning_rate": 0.05,
            "loss_function": "MultiClass",
            "eval_metric": "TotalF1",
            "random_seed": random_seed,
            "verbose": False
        }

        params.update(model_params)

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        y_pred = model.predict(X_test).ravel()

        raw_acc = accuracy_score(y_test, y_pred)

        theo_acc, numerator, denominator = theoretical_max_accuracy(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        fold_results.append({
            "psf_name": psf_name,
            "min_response_per_scenario": min_response_per_scenario,
            "fold": fold,

            # 전체 train 후보
            "n_train_all": len(train_df_all),
            "n_train_all_scenario": train_df_all["scenario_id"].nunique(),

            # 실제 학습에 사용된 train
            "n_train_used": len(train_df),
            "n_train_used_scenario": train_df["scenario_id"].nunique(),

            # 평가는 전체 test 사용
            "n_test": len(test_df),
            "n_test_scenario": test_df["scenario_id"].nunique(),

            # 참고용: test 안의 단일응답/2개이상 시나리오 수
            "n_test_single_response": (
                test_df["scenario_response_count"] == 1
            ).sum(),
            "n_test_multi_response": (
                test_df["scenario_response_count"] >= 2
            ).sum(),

            "raw_accuracy": raw_acc,
            "theoretical_accuracy": theo_acc,
            "numerator": numerator,
            "denominator": denominator,
            **model_params
        })

        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        all_scenario_id.extend(scenario_test.values)

    if len(fold_results) == 0:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "유효한 fold 없음",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    fold_df = pd.DataFrame(fold_results)

    total_theo_acc, total_num, total_den = theoretical_max_accuracy(
        y_true=np.array(all_y_true),
        y_pred=np.array(all_y_pred),
        scenario_id=np.array(all_scenario_id)
    )

    return {
        "psf_name": psf_name,
        "status": "success",
        "min_response_per_scenario": min_response_per_scenario,

        # 전체 평가 대상 기준
        "n_data_eval": len(df),
        "n_scenario_eval": df["scenario_id"].nunique(),

        # 실제 학습 가능 데이터 기준
        "n_data_train_candidate": len(
            df[df["scenario_response_count"] >= min_response_per_scenario]
        ),
        "n_scenario_train_candidate": df.loc[
            df["scenario_response_count"] >= min_response_per_scenario,
            "scenario_id"
        ].nunique(),

        "n_features": len(feature_cols),
        "feature_cols": feature_cols,
        "fold_results": fold_df,

        "raw_accuracy_mean": fold_df["raw_accuracy"].mean(),
        "theoretical_accuracy_total": total_theo_acc,
        "theoretical_numerator": total_num,
        "theoretical_denominator": total_den,

        "y_true": np.array(all_y_true),
        "y_pred": np.array(all_y_pred),
        **model_params
    }

In [ ]:
from itertools import product

param_grid = []

for depth, iterations, l2_leaf_reg, random_strength in product(
    [8, 9],
    [500],
    [20, 50, 100],
    [0]
):
    param_grid.append({
        "depth": depth,
        "iterations": iterations,
        "l2_leaf_reg": l2_leaf_reg,
        "random_strength": random_strength,
        "learning_rate": 0.05
    })

len(param_grid)

In [ ]:
quality = "95"

target_psfs = [
    "교육_및_훈련", "복잡성", "장비_및_기기"
]

min_response_list = [1]

psf_tuning_results = {}

for psf_name in target_psfs:

    if psf_name not in quality_psf_tables[quality]:
        print(f"{psf_name} 없음")
        continue

    df = quality_psf_tables[quality][psf_name]

    psf_tuning_results[psf_name] = {}

    for min_n in min_response_list:

        psf_tuning_results[psf_name][min_n] = []

        for i, model_params in enumerate(param_grid, start=1):

            print(f"\n[{psf_name}] min_n={min_n} / {i}/{len(param_grid)}")
            print(model_params)

            result = run_catboost_by_psf(
                psf_name=psf_name,
                df=df,
                min_response_per_scenario=min_n,
                model_params=model_params,
                n_splits=5,
                random_seed=42,
                target_col=target_col,
                drop_base_cols=drop_base_cols
            )

            psf_tuning_results[psf_name][min_n].append(result)

            if result["status"] == "success":
                print(
                    f"raw={result['raw_accuracy_mean']:.4f}, "
                    f"theo={result['theoretical_accuracy_total']:.4f}, "
                    f"eval_n={result['n_data_eval']}, "
                    f"eval_scenario={result['n_scenario_eval']}, "
                    f"train_n={result['n_data_train_candidate']}, "
                    f"train_scenario={result['n_scenario_train_candidate']}"
                )
            else:
                print(f"Skip: {result['reason']}")

In [ ]:
summary_rows = []

for psf_name, min_results in psf_tuning_results.items():

    for min_n, results in min_results.items():

        for result in results:

            row = {
                "PSF": psf_name,
                "min_response_per_scenario": min_n,
                "status": result["status"],
                "reason": result.get("reason", None),

                "n_data_eval": result.get("n_data_eval", np.nan),
                "n_scenario_eval": result.get("n_scenario_eval", np.nan),

                "n_data_train_candidate": result.get("n_data_train_candidate", np.nan),
                "n_scenario_train_candidate": result.get("n_scenario_train_candidate", np.nan),

                "n_features": result.get("n_features", np.nan),
                "raw_accuracy_mean": result.get("raw_accuracy_mean", np.nan),
                "theoretical_accuracy": result.get("theoretical_accuracy_total", np.nan),
                "theoretical_numerator": result.get("theoretical_numerator", np.nan),
                "theoretical_denominator": result.get("theoretical_denominator", np.nan),

                "depth": result.get("depth", np.nan),
                "iterations": result.get("iterations", np.nan),
                "learning_rate": result.get("learning_rate", np.nan),
                "l2_leaf_reg": result.get("l2_leaf_reg", np.nan),
                "random_strength": result.get("random_strength", np.nan),
            }

            summary_rows.append(row)

psf_tuning_summary = pd.DataFrame(summary_rows)

display(
    psf_tuning_summary.sort_values(
        ["PSF", "min_response_per_scenario", "theoretical_accuracy"],
        ascending=[True, True, False]
    )
)

## 3.4 PSF 병렬 튜닝 실행 및 summary 저장

`3.2`의 데이터 준비, 함수 정의, `param_grid` 셀까지 실행한 뒤 아래 셀을 실행한다. 기존 `3.2`의 긴 직렬 튜닝 셀은 건너뛰어도 된다. 전체 PSF를 병렬로 실행하고 진행률만 표시한 뒤 summary를 `result` 폴더에 저장한다.


In [ ]:
def get_feature_cols(df, target_col="answer", drop_base_cols=None):
    if drop_base_cols is None:
        drop_base_cols = []

    exclude_cols = set(drop_base_cols + [
        target_col,
        "scenario_response_count",
        "scenario_id"
    ])

    return [
        c for c in df.columns
        if c not in exclude_cols
    ]


def run_catboost_by_psf(
    psf_name,
    df,
    min_response_per_scenario=1,   # 학습에 사용할 최소 시나리오 응답 수
    model_params=None,
    n_splits=5,
    random_seed=42,
    target_col="answer",
    drop_base_cols=None
):
    df = df.copy()

    if drop_base_cols is None:
        drop_base_cols = []

    if model_params is None:
        model_params = {}

    df = df[df[target_col].notna()].copy()

    feature_cols = get_feature_cols(
        df=df,
        target_col=target_col,
        drop_base_cols=drop_base_cols
    )

    # 전체 데이터 기준 시나리오 응답 수 계산
    df = add_scenario_response_count(df, feature_cols)

    # 전체 데이터 기준 scenario_id 생성
    df["scenario_id"] = make_scenario_id(df, feature_cols)

    if len(df) < 30:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": f"평가 데이터 수 부족: {len(df)}",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    X = df[feature_cols]
    y = df[target_col]
    groups = df["scenario_id"]

    cat_features = list(X.columns)

    n_unique_scenarios = groups.nunique()
    n_splits = min(n_splits, n_unique_scenarios)

    if n_splits < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "GroupKFold 불가능",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    gkf = GroupKFold(n_splits=n_splits)

    fold_results = []
    all_y_true = []
    all_y_pred = []
    all_scenario_id = []

    for fold, (train_idx_all, test_idx) in enumerate(
        gkf.split(X, y, groups),
        start=1
    ):
        train_df_all = df.iloc[train_idx_all].copy()
        test_df = df.iloc[test_idx].copy()

        # 핵심 수정:
        # 학습 데이터만 min_response_per_scenario 조건 적용
        train_df = train_df_all[
            train_df_all["scenario_response_count"] >= min_response_per_scenario
        ].copy()

        if len(train_df) == 0:
            continue

        if train_df[target_col].nunique() < 2:
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[target_col]

        # 테스트는 단일응답 포함 전체 사용
        X_test = test_df[feature_cols]
        y_test = test_df[target_col]
        scenario_test = test_df["scenario_id"]

        params = {
            "iterations": 500,
            "depth": 4,
            "learning_rate": 0.05,
            "loss_function": "MultiClass",
            "eval_metric": "TotalF1",
            "random_seed": random_seed,
            "verbose": False
        }

        params.update(model_params)

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        y_pred = model.predict(X_test).ravel()

        raw_acc = accuracy_score(y_test, y_pred)

        theo_acc, numerator, denominator = theoretical_max_accuracy(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        fold_results.append({
            "psf_name": psf_name,
            "min_response_per_scenario": min_response_per_scenario,
            "fold": fold,

            # 전체 train 후보
            "n_train_all": len(train_df_all),
            "n_train_all_scenario": train_df_all["scenario_id"].nunique(),

            # 실제 학습에 사용된 train
            "n_train_used": len(train_df),
            "n_train_used_scenario": train_df["scenario_id"].nunique(),

            # 평가는 전체 test 사용
            "n_test": len(test_df),
            "n_test_scenario": test_df["scenario_id"].nunique(),

            # 참고용: test 안의 단일응답/2개이상 시나리오 수
            "n_test_single_response": (
                test_df["scenario_response_count"] == 1
            ).sum(),
            "n_test_multi_response": (
                test_df["scenario_response_count"] >= 2
            ).sum(),

            "raw_accuracy": raw_acc,
            "theoretical_accuracy": theo_acc,
            "numerator": numerator,
            "denominator": denominator,
            **model_params
        })

        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        all_scenario_id.extend(scenario_test.values)

    if len(fold_results) == 0:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "유효한 fold 없음",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    fold_df = pd.DataFrame(fold_results)

    total_theo_acc, total_num, total_den = theoretical_max_accuracy(
        y_true=np.array(all_y_true),
        y_pred=np.array(all_y_pred),
        scenario_id=np.array(all_scenario_id)
    )

    return {
        "psf_name": psf_name,
        "status": "success",
        "min_response_per_scenario": min_response_per_scenario,

        # 전체 평가 대상 기준
        "n_data_eval": len(df),
        "n_scenario_eval": df["scenario_id"].nunique(),

        # 실제 학습 가능 데이터 기준
        "n_data_train_candidate": len(
            df[df["scenario_response_count"] >= min_response_per_scenario]
        ),
        "n_scenario_train_candidate": df.loc[
            df["scenario_response_count"] >= min_response_per_scenario,
            "scenario_id"
        ].nunique(),

        "n_features": len(feature_cols),
        "feature_cols": feature_cols,
        "fold_results": fold_df,

        "raw_accuracy_mean": fold_df["raw_accuracy"].mean(),
        "theoretical_accuracy_total": total_theo_acc,
        "theoretical_numerator": total_num,
        "theoretical_denominator": total_den,

        "y_true": np.array(all_y_true),
        "y_pred": np.array(all_y_pred),
        **model_params
    }

In [ ]:
from itertools import product

param_grid = []

for grow_policy, max_leaves, depth, iterations, l2_leaf_reg, random_strength in product(
    ["Lossguide"],
    [16, 32, 64],
    [8],
    [500],
    [20, 50, 100],
    [0, 5, 10]
):
    param_grid.append({
        "grow_policy": grow_policy,
        "max_leaves": max_leaves,
        "depth": depth,
        "iterations": iterations,
        "l2_leaf_reg": l2_leaf_reg,
        "random_strength": random_strength,
        "learning_rate": 0.05,
    })

len(param_grid)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from IPython.display import clear_output
import os

quality = "95"

# 기본값: 교육_및_훈련을 포함한 전체 PSF
# 특정 PSF만 돌리고 싶으면 예: ["복잡성", "의사소통"] 형태로 바꾼다.
# target_psfs_3_3 = list(quality_psf_tables[quality].keys())
target_psfs_3_3 = ["복잡성", "교육_및_훈련", "장비_및_기기"]

min_response_list_3_3 = [1, 2]
max_workers_3_3 = min(4, os.cpu_count() or 1)
catboost_thread_count_3_3 = 1
keep_detail_results_3_3 = False

result_dir_3_3 = Path("result")
result_dir_3_3.mkdir(exist_ok=True)

parallel_tasks_3_3 = []
for psf_name in target_psfs_3_3:
    if psf_name not in quality_psf_tables[quality]:
        continue

    df = quality_psf_tables[quality][psf_name]

    for min_n in min_response_list_3_3:
        for grid_index, model_params in enumerate(param_grid, start=1):
            parallel_tasks_3_3.append({
                "psf_name": psf_name,
                "df": df,
                "min_n": min_n,
                "grid_index": grid_index,
                "model_params": {
                    **model_params,
                    "thread_count": catboost_thread_count_3_3,
                },
            })

psf_tuning_results_3_3 = {
    psf_name: {
        min_n: [None] * len(param_grid)
        for min_n in min_response_list_3_3
    }
    for psf_name in target_psfs_3_3
    if psf_name in quality_psf_tables[quality]
}

total_tasks_3_3 = len(parallel_tasks_3_3)
completed_tasks_3_3 = 0
print(f"3.3 병렬 튜닝 진행률: 0.0% (0/{total_tasks_3_3})")


def run_parallel_tuning_task_3_3(task):
    result = run_catboost_by_psf(
        psf_name=task["psf_name"],
        df=task["df"],
        min_response_per_scenario=task["min_n"],
        model_params=task["model_params"],
        n_splits=5,
        random_seed=42,
        target_col=target_col,
        drop_base_cols=drop_base_cols,
    )

    result["grid_index"] = task["grid_index"]

    if not keep_detail_results_3_3:
        result.pop("feature_cols", None)
        result.pop("fold_results", None)
        result.pop("y_true", None)
        result.pop("y_pred", None)

    return task["psf_name"], task["min_n"], task["grid_index"], result


with ThreadPoolExecutor(max_workers=max_workers_3_3) as executor:
    future_to_task_3_3 = {
        executor.submit(run_parallel_tuning_task_3_3, task): task
        for task in parallel_tasks_3_3
    }

    for future in as_completed(future_to_task_3_3):
        task = future_to_task_3_3[future]
        psf_name = task["psf_name"]
        min_n = task["min_n"]
        grid_index = task["grid_index"]

        try:
            _, _, _, result = future.result()
        except Exception as exc:
            result = {
                "psf_name": psf_name,
                "status": "error",
                "reason": repr(exc),
                "min_response_per_scenario": min_n,
                "grid_index": grid_index,
                **task["model_params"],
            }

        psf_tuning_results_3_3[psf_name][min_n][grid_index - 1] = result

        completed_tasks_3_3 += 1
        progress_pct_3_3 = (
            100 * completed_tasks_3_3 / total_tasks_3_3
            if total_tasks_3_3
            else 100
        )
        clear_output(wait=True)
        print(
            f"3.3 병렬 튜닝 진행률: "
            f"{progress_pct_3_3:.1f}% "
            f"({completed_tasks_3_3}/{total_tasks_3_3})"
        )

summary_rows_3_3 = []
param_columns_3_3 = sorted({
    key
    for model_params in param_grid
    for key in {**model_params, "thread_count": catboost_thread_count_3_3}.keys()
})

for psf_name, min_results in psf_tuning_results_3_3.items():
    for min_n, results in min_results.items():
        for grid_index, result in enumerate(results, start=1):
            if result is None:
                result = {
                    "psf_name": psf_name,
                    "status": "error",
                    "reason": "작업 결과 없음",
                    "min_response_per_scenario": min_n,
                    "grid_index": grid_index,
                }

            row = {
                "PSF": psf_name,
                "quality": quality,
                "min_response_per_scenario": min_n,
                "grid_index": grid_index,
                "status": result.get("status"),
                "reason": result.get("reason", None),
                "n_data": result.get("n_data", np.nan),
                "n_scenario": result.get("n_scenario", np.nan),
                "n_features": result.get("n_features", np.nan),
                "raw_accuracy_mean": result.get("raw_accuracy_mean", np.nan),
                "theoretical_accuracy": result.get("theoretical_accuracy_total", np.nan),
                "theoretical_numerator": result.get("theoretical_numerator", np.nan),
                "theoretical_denominator": result.get("theoretical_denominator", np.nan),
            }

            for param_name in param_columns_3_3:
                row[param_name] = result.get(param_name, np.nan)

            summary_rows_3_3.append(row)

psf_tuning_summary_3_3 = pd.DataFrame(summary_rows_3_3)

if not psf_tuning_summary_3_3.empty:
    psf_tuning_summary_3_3 = psf_tuning_summary_3_3.sort_values(
        ["PSF", "min_response_per_scenario", "theoretical_accuracy"],
        ascending=[True, True, False],
        na_position="last",
    ).reset_index(drop=True)

timestamp_3_3 = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
summary_csv_path_3_3 = result_dir_3_3 / f"psf_parallel_tuning_summary_q{quality}_{timestamp_3_3}.csv"
summary_xlsx_path_3_3 = result_dir_3_3 / f"psf_parallel_tuning_summary_q{quality}_{timestamp_3_3}.xlsx"
latest_csv_path_3_3 = result_dir_3_3 / "psf_parallel_tuning_summary_latest.csv"
latest_xlsx_path_3_3 = result_dir_3_3 / "psf_parallel_tuning_summary_latest.xlsx"

psf_tuning_summary_3_3.to_csv(summary_csv_path_3_3, index=False, encoding="utf-8-sig")
psf_tuning_summary_3_3.to_csv(latest_csv_path_3_3, index=False, encoding="utf-8-sig")

try:
    psf_tuning_summary_3_3.to_excel(summary_xlsx_path_3_3, index=False)
    psf_tuning_summary_3_3.to_excel(latest_xlsx_path_3_3, index=False)
except Exception:
    pass

clear_output(wait=True)
print(f"3.3 병렬 튜닝 완료: 100.0% ({completed_tasks_3_3}/{total_tasks_3_3})")
print(f"CSV 저장: {summary_csv_path_3_3}")
print(f"최신 CSV 저장: {latest_csv_path_3_3}")


## 3.5 scenario label groupkfold 적용

In [ ]:
def scenario_label_kfold_split(
    df,
    scenario_col="scenario_id",
    target_col="answer",
    n_splits=5,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)
    fold_indices = [[] for _ in range(n_splits)]

    temp_df = df[[scenario_col, target_col]].reset_index(drop=True)

    for _, scenario_df in temp_df.groupby(scenario_col):
        for _, label_df in scenario_df.groupby(target_col):
            idx = label_df.index.to_numpy().copy()  # 핵심 수정
            rng.shuffle(idx)

            for i, row_idx in enumerate(idx):
                fold_indices[i % n_splits].append(row_idx)

    all_indices = np.arange(len(temp_df))

    for fold in range(n_splits):
        test_idx = np.array(fold_indices[fold], dtype=int)

        if len(test_idx) == 0:
            continue

        train_idx = np.setdiff1d(all_indices, test_idx)

        yield train_idx, test_idx

In [ ]:
def get_feature_cols(df, target_col="answer", drop_base_cols=None):
    if drop_base_cols is None:
        drop_base_cols = []

    exclude_cols = set(drop_base_cols + [
        target_col,
        "scenario_response_count",
        "scenario_id"
    ])

    return [
        c for c in df.columns
        if c not in exclude_cols
    ]


def run_catboost_by_psf(
    psf_name,
    df,
    min_response_per_scenario=1,   # 학습에 사용할 최소 시나리오 응답 수
    model_params=None,
    n_splits=5,
    random_seed=42,
    target_col="answer",
    drop_base_cols=None
):
    df = df.copy()

    if drop_base_cols is None:
        drop_base_cols = []

    if model_params is None:
        model_params = {}

    df = df[df[target_col].notna()].copy()

    feature_cols = get_feature_cols(
        df=df,
        target_col=target_col,
        drop_base_cols=drop_base_cols
    )

    # 전체 데이터 기준 시나리오 응답 수 계산
    df = add_scenario_response_count(df, feature_cols)

    # 전체 데이터 기준 scenario_id 생성
    df["scenario_id"] = make_scenario_id(df, feature_cols)

    if len(df) < 30:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": f"평가 데이터 수 부족: {len(df)}",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    X = df[feature_cols]
    y = df[target_col]
    groups = df["scenario_id"]

    cat_features = list(X.columns)

    n_unique_scenarios = groups.nunique()
    n_splits = min(n_splits, len(df))

    if n_splits < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "KFold 불가능",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    fold_results = []
    all_y_true = []
    all_y_pred = []
    all_scenario_id = []

    split_iter = scenario_label_kfold_split(
        df=df,
        scenario_col="scenario_id",
        target_col=target_col,
        n_splits=n_splits,
        random_seed=random_seed
    )

    for fold, (train_idx_all, test_idx) in enumerate(
        split_iter,
        start=1
    ):
        train_df_all = df.loc[train_idx_all].copy()
        test_df = df.loc[test_idx].copy()

        # 핵심 수정:
        # 학습 데이터만 min_response_per_scenario 조건 적용
        train_df = train_df_all[
            train_df_all["scenario_response_count"] >= min_response_per_scenario
        ].copy()

        if len(train_df) == 0:
            continue

        if train_df[target_col].nunique() < 2:
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[target_col]

        # 테스트는 단일응답 포함 전체 사용
        X_test = test_df[feature_cols]
        y_test = test_df[target_col]
        scenario_test = test_df["scenario_id"]

        params = {
            "iterations": 500,
            "depth": 4,
            "learning_rate": 0.05,
            "loss_function": "MultiClass",
            "eval_metric": "TotalF1",
            "random_seed": random_seed,
            "verbose": False
        }

        params.update(model_params)

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        y_pred = model.predict(X_test).ravel()

        raw_acc = accuracy_score(y_test, y_pred)

        theo_acc, numerator, denominator = theoretical_max_accuracy(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        fold_results.append({
            "psf_name": psf_name,
            "min_response_per_scenario": min_response_per_scenario,
            "fold": fold,

            # 전체 train 후보
            "n_train_all": len(train_df_all),
            "n_train_all_scenario": train_df_all["scenario_id"].nunique(),

            # 실제 학습에 사용된 train
            "n_train_used": len(train_df),
            "n_train_used_scenario": train_df["scenario_id"].nunique(),

            # 평가는 전체 test 사용
            "n_test": len(test_df),
            "n_test_scenario": test_df["scenario_id"].nunique(),

            # 참고용: test 안의 단일응답/2개이상 시나리오 수
            "n_test_single_response": (
                test_df["scenario_response_count"] == 1
            ).sum(),
            "n_test_multi_response": (
                test_df["scenario_response_count"] >= 2
            ).sum(),

            "raw_accuracy": raw_acc,
            "theoretical_accuracy": theo_acc,
            "numerator": numerator,
            "denominator": denominator,
            **model_params
        })

        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        all_scenario_id.extend(scenario_test.values)

    if len(fold_results) == 0:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "유효한 fold 없음",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    fold_df = pd.DataFrame(fold_results)

    total_theo_acc, total_num, total_den = theoretical_max_accuracy(
        y_true=np.array(all_y_true),
        y_pred=np.array(all_y_pred),
        scenario_id=np.array(all_scenario_id)
    )

    return {
        "psf_name": psf_name,
        "status": "success",
        "min_response_per_scenario": min_response_per_scenario,

        # 전체 평가 대상 기준
        "n_data_eval": len(df),
        "n_scenario_eval": df["scenario_id"].nunique(),

        # 실제 학습 가능 데이터 기준
        "n_data_train_candidate": len(
            df[df["scenario_response_count"] >= min_response_per_scenario]
        ),
        "n_scenario_train_candidate": df.loc[
            df["scenario_response_count"] >= min_response_per_scenario,
            "scenario_id"
        ].nunique(),

        "n_features": len(feature_cols),
        "feature_cols": feature_cols,
        "fold_results": fold_df,

        "raw_accuracy_mean": fold_df["raw_accuracy"].mean(),
        "theoretical_accuracy_total": total_theo_acc,
        "theoretical_numerator": total_num,
        "theoretical_denominator": total_den,

        "y_true": np.array(all_y_true),
        "y_pred": np.array(all_y_pred),
        **model_params
    }

In [ ]:
from itertools import product

param_grid = []

for grow_policy, max_leaves, depth, iterations, l2_leaf_reg, random_strength in product(
    ["Lossguide"],
    [32, 64],
    [8],
    [500, 1000],
    [20],
    [1, 5, 10]
):
    param_grid.append({
        "grow_policy": grow_policy,
        "max_leaves": max_leaves,
        "depth": depth,
        "iterations": iterations,
        "l2_leaf_reg": l2_leaf_reg,
        "random_strength": random_strength,
        "learning_rate": 0.05,
    })

len(param_grid)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from IPython.display import clear_output
import os

quality = "95"

# 기본값: 교육_및_훈련을 포함한 전체 PSF
# 특정 PSF만 돌리고 싶으면 예: ["복잡성", "의사소통"] 형태로 바꾼다.
# target_psfs_3_3 = list(quality_psf_tables[quality].keys())
target_psfs_3_3 = ["복잡성", "교육_및_훈련", "장비_및_기기"]

min_response_list_3_3 = [1, 2]
max_workers_3_3 = min(4, os.cpu_count() or 1)
catboost_thread_count_3_3 = 1
keep_detail_results_3_3 = False

result_dir_3_3 = Path("result")
result_dir_3_3.mkdir(exist_ok=True)

parallel_tasks_3_3 = []
for psf_name in target_psfs_3_3:
    if psf_name not in quality_psf_tables[quality]:
        continue

    df = quality_psf_tables[quality][psf_name]

    for min_n in min_response_list_3_3:
        for grid_index, model_params in enumerate(param_grid, start=1):
            parallel_tasks_3_3.append({
                "psf_name": psf_name,
                "df": df,
                "min_n": min_n,
                "grid_index": grid_index,
                "model_params": {
                    **model_params,
                    "thread_count": catboost_thread_count_3_3,
                },
            })

psf_tuning_results_3_3 = {
    psf_name: {
        min_n: [None] * len(param_grid)
        for min_n in min_response_list_3_3
    }
    for psf_name in target_psfs_3_3
    if psf_name in quality_psf_tables[quality]
}

total_tasks_3_3 = len(parallel_tasks_3_3)
completed_tasks_3_3 = 0
print(f"3.3 병렬 튜닝 진행률: 0.0% (0/{total_tasks_3_3})")


def run_parallel_tuning_task_3_3(task):
    result = run_catboost_by_psf(
        psf_name=task["psf_name"],
        df=task["df"],
        min_response_per_scenario=task["min_n"],
        model_params=task["model_params"],
        n_splits=5,
        random_seed=42,
        target_col=target_col,
        drop_base_cols=drop_base_cols,
    )

    result["grid_index"] = task["grid_index"]

    if not keep_detail_results_3_3:
        result.pop("feature_cols", None)
        result.pop("fold_results", None)
        result.pop("y_true", None)
        result.pop("y_pred", None)

    return task["psf_name"], task["min_n"], task["grid_index"], result


with ThreadPoolExecutor(max_workers=max_workers_3_3) as executor:
    future_to_task_3_3 = {
        executor.submit(run_parallel_tuning_task_3_3, task): task
        for task in parallel_tasks_3_3
    }

    for future in as_completed(future_to_task_3_3):
        task = future_to_task_3_3[future]
        psf_name = task["psf_name"]
        min_n = task["min_n"]
        grid_index = task["grid_index"]

        try:
            _, _, _, result = future.result()
        except Exception as exc:
            result = {
                "psf_name": psf_name,
                "status": "error",
                "reason": repr(exc),
                "min_response_per_scenario": min_n,
                "grid_index": grid_index,
                **task["model_params"],
            }

        psf_tuning_results_3_3[psf_name][min_n][grid_index - 1] = result

        completed_tasks_3_3 += 1
        progress_pct_3_3 = (
            100 * completed_tasks_3_3 / total_tasks_3_3
            if total_tasks_3_3
            else 100
        )
        clear_output(wait=True)
        print(
            f"3.3 병렬 튜닝 진행률: "
            f"{progress_pct_3_3:.1f}% "
            f"({completed_tasks_3_3}/{total_tasks_3_3})"
        )

summary_rows_3_3 = []
param_columns_3_3 = sorted({
    key
    for model_params in param_grid
    for key in {**model_params, "thread_count": catboost_thread_count_3_3}.keys()
})

for psf_name, min_results in psf_tuning_results_3_3.items():
    for min_n, results in min_results.items():
        for grid_index, result in enumerate(results, start=1):
            if result is None:
                result = {
                    "psf_name": psf_name,
                    "status": "error",
                    "reason": "작업 결과 없음",
                    "min_response_per_scenario": min_n,
                    "grid_index": grid_index,
                }

            row = {
                "PSF": psf_name,
                "quality": quality,
                "min_response_per_scenario": min_n,
                "grid_index": grid_index,
                "status": result.get("status"),
                "reason": result.get("reason", None),
                "n_data": result.get("n_data", np.nan),
                "n_scenario": result.get("n_scenario", np.nan),
                "n_features": result.get("n_features", np.nan),
                "raw_accuracy_mean": result.get("raw_accuracy_mean", np.nan),
                "theoretical_accuracy": result.get("theoretical_accuracy_total", np.nan),
                "theoretical_numerator": result.get("theoretical_numerator", np.nan),
                "theoretical_denominator": result.get("theoretical_denominator", np.nan),
            }

            for param_name in param_columns_3_3:
                row[param_name] = result.get(param_name, np.nan)

            summary_rows_3_3.append(row)

psf_tuning_summary_3_3 = pd.DataFrame(summary_rows_3_3)

if not psf_tuning_summary_3_3.empty:
    psf_tuning_summary_3_3 = psf_tuning_summary_3_3.sort_values(
        ["PSF", "min_response_per_scenario", "theoretical_accuracy"],
        ascending=[True, True, False],
        na_position="last",
    ).reset_index(drop=True)

timestamp_3_3 = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
summary_csv_path_3_3 = result_dir_3_3 / f"psf_parallel_tuning_summary_q{quality}_{timestamp_3_3}.csv"
summary_xlsx_path_3_3 = result_dir_3_3 / f"psf_parallel_tuning_summary_q{quality}_{timestamp_3_3}.xlsx"
latest_csv_path_3_3 = result_dir_3_3 / "psf_parallel_tuning_summary_latest.csv"
latest_xlsx_path_3_3 = result_dir_3_3 / "psf_parallel_tuning_summary_latest.xlsx"

psf_tuning_summary_3_3.to_csv(summary_csv_path_3_3, index=False, encoding="utf-8-sig")
psf_tuning_summary_3_3.to_csv(latest_csv_path_3_3, index=False, encoding="utf-8-sig")

try:
    psf_tuning_summary_3_3.to_excel(summary_xlsx_path_3_3, index=False)
    psf_tuning_summary_3_3.to_excel(latest_xlsx_path_3_3, index=False)
except Exception:
    pass

clear_output(wait=True)
print(f"3.3 병렬 튜닝 완료: 100.0% ({completed_tasks_3_3}/{total_tasks_3_3})")
print(f"CSV 저장: {summary_csv_path_3_3}")
print(f"최신 CSV 저장: {latest_csv_path_3_3}")


# 4. 분포기반 예측

In [ ]:
def scenario_label_kfold_split(
    df,
    scenario_col="scenario_id",
    target_col="answer",
    n_splits=5,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)
    fold_indices = [[] for _ in range(n_splits)]

    temp_df = df[[scenario_col, target_col]].reset_index(drop=True)

    for _, scenario_df in temp_df.groupby(scenario_col):
        for _, label_df in scenario_df.groupby(target_col):
            idx = label_df.index.to_numpy().copy()  # 핵심 수정
            rng.shuffle(idx)

            for i, row_idx in enumerate(idx):
                fold_indices[i % n_splits].append(row_idx)

    all_indices = np.arange(len(temp_df))

    for fold in range(n_splits):
        test_idx = np.array(fold_indices[fold], dtype=int)

        if len(test_idx) == 0:
            continue

        train_idx = np.setdiff1d(all_indices, test_idx)

        yield train_idx, test_idx

In [ ]:
def distribution_overlap_by_scenario(
    y_true,
    y_pred,
    scenario_id
):
    eval_df = pd.DataFrame({
        "scenario_id": scenario_id,
        "y_true": y_true,
        "y_pred": y_pred
    })

    rows = []

    for sid, g in eval_df.groupby("scenario_id"):
        true_dist = g["y_true"].value_counts(normalize=True)
        pred_dist = g["y_pred"].value_counts(normalize=True)

        labels = sorted(set(true_dist.index) | set(pred_dist.index))

        overlap = sum(
            min(true_dist.get(label, 0), pred_dist.get(label, 0))
            for label in labels
        )

        rows.append({
            "scenario_id": sid,
            "n_response": len(g),
            "distribution_overlap": overlap
        })

    overlap_df = pd.DataFrame(rows)

    # 시나리오 기준 평균: 모든 시나리오 동일 가중치
    scenario_mean = overlap_df["distribution_overlap"].mean()

    # 응답 기준 평균: 응답 많은 시나리오에 더 큰 가중치
    response_weighted_mean = np.average(
        overlap_df["distribution_overlap"],
        weights=overlap_df["n_response"]
    )

    return scenario_mean, response_weighted_mean, overlap_df

In [ ]:
def get_feature_cols(df, target_col="answer", drop_base_cols=None):
    if drop_base_cols is None:
        drop_base_cols = []

    exclude_cols = set(drop_base_cols + [
        target_col,
        "scenario_response_count",
        "scenario_id"
    ])

    return [
        c for c in df.columns
        if c not in exclude_cols
    ]


def run_catboost_by_psf(
    psf_name,
    df,
    min_response_per_scenario=1,   # 학습에 사용할 최소 시나리오 응답 수
    model_params=None,
    n_splits=5,
    random_seed=42,
    target_col="answer",
    drop_base_cols=None
):
    df = df.copy()

    if drop_base_cols is None:
        drop_base_cols = []

    if model_params is None:
        model_params = {}

    df = df[df[target_col].notna()].copy()

    feature_cols = get_feature_cols(
        df=df,
        target_col=target_col,
        drop_base_cols=drop_base_cols
    )

    # 전체 데이터 기준 시나리오 응답 수 계산
    df = add_scenario_response_count(df, feature_cols)

    # 전체 데이터 기준 scenario_id 생성
    df["scenario_id"] = make_scenario_id(df, feature_cols)

    if len(df) < 30:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": f"평가 데이터 수 부족: {len(df)}",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    X = df[feature_cols]
    y = df[target_col]
    groups = df["scenario_id"]

    cat_features = list(X.columns)

    n_unique_scenarios = groups.nunique()
    n_splits = min(n_splits, len(df))

    if n_splits < 2:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "KFold 불가능",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    fold_results = []
    all_y_true = []
    all_y_pred = []
    all_scenario_id = []

    split_iter = scenario_label_kfold_split(
        df=df,
        scenario_col="scenario_id",
        target_col=target_col,
        n_splits=n_splits,
        random_seed=random_seed
    )

    for fold, (train_idx_all, test_idx) in enumerate(
        split_iter,
        start=1
    ):
        train_df_all = df.loc[train_idx_all].copy()
        test_df = df.loc[test_idx].copy()

        # 핵심 수정:
        # 학습 데이터만 min_response_per_scenario 조건 적용
        train_df = train_df_all[
            train_df_all["scenario_response_count"] >= min_response_per_scenario
        ].copy()

        if len(train_df) == 0:
            continue

        if train_df[target_col].nunique() < 2:
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[target_col]

        # 테스트는 단일응답 포함 전체 사용
        X_test = test_df[feature_cols]
        y_test = test_df[target_col]
        scenario_test = test_df["scenario_id"]

        params = {
            "iterations": 500,
            "depth": 4,
            "learning_rate": 0.05,
            "loss_function": "MultiClass",
            "eval_metric": "TotalF1",
            "random_seed": random_seed,
            "verbose": False
        }

        params.update(model_params)

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        y_pred = model.predict(X_test).ravel()

        raw_acc = accuracy_score(y_test, y_pred)

        theo_acc, numerator, denominator = theoretical_max_accuracy(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        do_scenario_mean, do_response_weighted, do_detail = distribution_overlap_by_scenario(
            y_true=y_test.values,
            y_pred=y_pred,
            scenario_id=scenario_test.values
        )

        fold_results.append({
            "psf_name": psf_name,
            "min_response_per_scenario": min_response_per_scenario,
            "fold": fold,

            # 전체 train 후보
            "n_train_all": len(train_df_all),
            "n_train_all_scenario": train_df_all["scenario_id"].nunique(),

            # 실제 학습에 사용된 train
            "n_train_used": len(train_df),
            "n_train_used_scenario": train_df["scenario_id"].nunique(),

            # 평가는 전체 test 사용
            "n_test": len(test_df),
            "n_test_scenario": test_df["scenario_id"].nunique(),

            # 참고용: test 안의 단일응답/2개이상 시나리오 수
            "n_test_single_response": (
                test_df["scenario_response_count"] == 1
            ).sum(),
            "n_test_multi_response": (
                test_df["scenario_response_count"] >= 2
            ).sum(),

            "raw_accuracy": raw_acc,
            "theoretical_accuracy": theo_acc,
            "numerator": numerator,
            "denominator": denominator,
            "distribution_overlap_scenario_mean": do_scenario_mean,
            "distribution_overlap_response_weighted": do_response_weighted,
            **model_params
        })

        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        all_scenario_id.extend(scenario_test.values)

    if len(fold_results) == 0:
        return {
            "psf_name": psf_name,
            "status": "skip",
            "reason": "유효한 fold 없음",
            "min_response_per_scenario": min_response_per_scenario,
            **model_params
        }

    fold_df = pd.DataFrame(fold_results)

    total_theo_acc, total_num, total_den = theoretical_max_accuracy(
        y_true=np.array(all_y_true),
        y_pred=np.array(all_y_pred),
        scenario_id=np.array(all_scenario_id)
    )

    total_do_scenario_mean, total_do_response_weighted, total_do_detail = (
        distribution_overlap_by_scenario(
            y_true=np.array(all_y_true),
            y_pred=np.array(all_y_pred),
            scenario_id=np.array(all_scenario_id)
        )
    )

    return {
        "psf_name": psf_name,
        "status": "success",
        "min_response_per_scenario": min_response_per_scenario,

        # 전체 평가 대상 기준
        "n_data_eval": len(df),
        "n_scenario_eval": df["scenario_id"].nunique(),

        # 실제 학습 가능 데이터 기준
        "n_data_train_candidate": len(
            df[df["scenario_response_count"] >= min_response_per_scenario]
        ),
        "n_scenario_train_candidate": df.loc[
            df["scenario_response_count"] >= min_response_per_scenario,
            "scenario_id"
        ].nunique(),

        "n_features": len(feature_cols),
        "feature_cols": feature_cols,
        "fold_results": fold_df,

        "raw_accuracy_mean": fold_df["raw_accuracy"].mean(),
        "theoretical_accuracy_total": total_theo_acc,
        "theoretical_numerator": total_num,
        "theoretical_denominator": total_den,
        "distribution_overlap_scenario_mean": total_do_scenario_mean,
        "distribution_overlap_response_weighted": total_do_response_weighted,
        "distribution_overlap_detail": total_do_detail,

        "y_true": np.array(all_y_true),
        "y_pred": np.array(all_y_pred),
        **model_params
    }

In [ ]:
from itertools import product

param_grid = []

for grow_policy, max_leaves, depth, iterations, l2_leaf_reg, random_strength in product(
    ["Lossguide"],
    [16],
    [8],
    [500],
    [20],
    [1]
):
    param_grid.append({
        "grow_policy": grow_policy,
        "max_leaves": max_leaves,
        "depth": depth,
        "iterations": iterations,
        "l2_leaf_reg": l2_leaf_reg,
        "random_strength": random_strength,
        "learning_rate": 0.05,
    })

len(param_grid)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from IPython.display import clear_output
import os

quality = "95"

# 기본값: 교육_및_훈련을 포함한 전체 PSF
# 특정 PSF만 돌리고 싶으면 예: ["복잡성", "의사소통"] 형태로 바꾼다.
# target_psfs_3_3 = list(quality_psf_tables[quality].keys())
target_psfs_3_3 = ["복잡성", "교육_및_훈련", "장비_및_기기"]

min_response_list_3_3 = [1]
max_workers_3_3 = min(4, os.cpu_count() or 1)
catboost_thread_count_3_3 = 1
keep_detail_results_3_3 = False

result_dir_3_3 = Path("result")
result_dir_3_3.mkdir(exist_ok=True)

parallel_tasks_3_3 = []
for psf_name in target_psfs_3_3:
    if psf_name not in quality_psf_tables[quality]:
        continue

    df = quality_psf_tables[quality][psf_name]

    for min_n in min_response_list_3_3:
        for grid_index, model_params in enumerate(param_grid, start=1):
            parallel_tasks_3_3.append({
                "psf_name": psf_name,
                "df": df,
                "min_n": min_n,
                "grid_index": grid_index,
                "model_params": {
                    **model_params,
                    "thread_count": catboost_thread_count_3_3,
                },
            })

psf_tuning_results_3_3 = {
    psf_name: {
        min_n: [None] * len(param_grid)
        for min_n in min_response_list_3_3
    }
    for psf_name in target_psfs_3_3
    if psf_name in quality_psf_tables[quality]
}

total_tasks_3_3 = len(parallel_tasks_3_3)
completed_tasks_3_3 = 0
print(f"3.3 병렬 튜닝 진행률: 0.0% (0/{total_tasks_3_3})")


def run_parallel_tuning_task_3_3(task):
    result = run_catboost_by_psf(
        psf_name=task["psf_name"],
        df=task["df"],
        min_response_per_scenario=task["min_n"],
        model_params=task["model_params"],
        n_splits=5,
        random_seed=42,
        target_col=target_col,
        drop_base_cols=drop_base_cols,
    )

    result["grid_index"] = task["grid_index"]

    if not keep_detail_results_3_3:
        result.pop("feature_cols", None)
        result.pop("fold_results", None)
        result.pop("y_true", None)
        result.pop("y_pred", None)

    return task["psf_name"], task["min_n"], task["grid_index"], result


with ThreadPoolExecutor(max_workers=max_workers_3_3) as executor:
    future_to_task_3_3 = {
        executor.submit(run_parallel_tuning_task_3_3, task): task
        for task in parallel_tasks_3_3
    }

    for future in as_completed(future_to_task_3_3):
        task = future_to_task_3_3[future]
        psf_name = task["psf_name"]
        min_n = task["min_n"]
        grid_index = task["grid_index"]

        try:
            _, _, _, result = future.result()
        except Exception as exc:
            result = {
                "psf_name": psf_name,
                "status": "error",
                "reason": repr(exc),
                "min_response_per_scenario": min_n,
                "grid_index": grid_index,
                **task["model_params"],
            }

        psf_tuning_results_3_3[psf_name][min_n][grid_index - 1] = result

        completed_tasks_3_3 += 1
        progress_pct_3_3 = (
            100 * completed_tasks_3_3 / total_tasks_3_3
            if total_tasks_3_3
            else 100
        )
        clear_output(wait=True)
        print(
            f"3.3 병렬 튜닝 진행률: "
            f"{progress_pct_3_3:.1f}% "
            f"({completed_tasks_3_3}/{total_tasks_3_3})"
        )

summary_rows_3_3 = []
param_columns_3_3 = sorted({
    key
    for model_params in param_grid
    for key in {**model_params, "thread_count": catboost_thread_count_3_3}.keys()
})

for psf_name, min_results in psf_tuning_results_3_3.items():
    for min_n, results in min_results.items():
        for grid_index, result in enumerate(results, start=1):
            if result is None:
                result = {
                    "psf_name": psf_name,
                    "status": "error",
                    "reason": "작업 결과 없음",
                    "min_response_per_scenario": min_n,
                    "grid_index": grid_index,
                }

            row = {
                "PSF": psf_name,
                "quality": quality,
                "min_response_per_scenario": min_n,
                "grid_index": grid_index,
                "status": result.get("status"),
                "reason": result.get("reason", None),
                "n_data": result.get("n_data", np.nan),
                "n_scenario": result.get("n_scenario", np.nan),
                "n_features": result.get("n_features", np.nan),
                "raw_accuracy_mean": result.get("raw_accuracy_mean", np.nan),
                "theoretical_accuracy": result.get("theoretical_accuracy_total", np.nan),
                "theoretical_numerator": result.get("theoretical_numerator", np.nan),
                "theoretical_denominator": result.get("theoretical_denominator", np.nan),
                "distribution_overlap_scenario_mean": result.get(
                    "distribution_overlap_scenario_mean", np.nan
                ),
                "distribution_overlap_response_weighted": result.get(
                    "distribution_overlap_response_weighted", np.nan
                ),
            }

            for param_name in param_columns_3_3:
                row[param_name] = result.get(param_name, np.nan)

            summary_rows_3_3.append(row)

psf_tuning_summary_3_3 = pd.DataFrame(summary_rows_3_3)

if not psf_tuning_summary_3_3.empty:
    psf_tuning_summary_3_3 = psf_tuning_summary_3_3.sort_values(
        ["PSF", "min_response_per_scenario", "theoretical_accuracy"],
        ascending=[True, True, False],
        na_position="last",
    ).reset_index(drop=True)

timestamp_3_3 = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
summary_xlsx_path_3_3 = result_dir_3_3 / f"psf_parallel_tuning_summary_q{quality}_{timestamp_3_3}.xlsx"
latest_xlsx_path_3_3 = result_dir_3_3 / "psf_parallel_tuning_summary_latest.xlsx"

psf_tuning_summary_3_3.to_csv(summary_csv_path_3_3, index=False, encoding="utf-8-sig")
psf_tuning_summary_3_3.to_csv(latest_csv_path_3_3, index=False, encoding="utf-8-sig")

try:
    psf_tuning_summary_3_3.to_excel(summary_xlsx_path_3_3, index=False)
    psf_tuning_summary_3_3.to_excel(latest_xlsx_path_3_3, index=False)
except Exception:
    pass

clear_output(wait=True)
print(f"3.3 병렬 튜닝 완료: 100.0% ({completed_tasks_3_3}/{total_tasks_3_3})")
print(f"CSV 저장: {summary_csv_path_3_3}")
print(f"최신 CSV 저장: {latest_csv_path_3_3}")


# X1

In [ ]:
from functools import reduce

join_keys = ["empid", "survey_year", "survey_id", "set_id", "repeat_no"]
quality_cols = ["quality_100", "quality_95", "quality_90"]

missing_key_tables = {
    psf: [c for c in join_keys if c not in table.columns]
    for psf, table in psf_tables.items()
    if any(c not in table.columns for c in join_keys)
}

if missing_key_tables:
    raise ValueError(
        "PSF 테이블에 set_id/repeat_no가 없습니다. "
        "preprocess_v1.2에서 PSF 테이블을 다시 생성한 뒤 실행하세요. "
        f"missing={missing_key_tables}"
    )

wide_tables = []

for psf, table in psf_tables.items():
    temp = table.copy()

    answer_col = f"{psf}_answer"
    temp = temp.rename(columns={"answer": answer_col})

    metadata_cols = join_keys + ["PSF_name", answer_col] + quality_cols
    feature_cols = [c for c in temp.columns if c not in metadata_cols]

    rename_cols = {
        c: f"{psf}_{c}"
        for c in feature_cols + quality_cols
        if c in temp.columns
    }

    temp = temp.rename(columns=rename_cols)

    keep_cols = (
        join_keys
        + [answer_col]
        + [rename_cols[c] for c in quality_cols if c in rename_cols]
        + [rename_cols[c] for c in feature_cols if c in rename_cols]
    )

    wide_tables.append(temp[keep_cols])

all_psf_wide = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=join_keys,
        how="outer"
    ),
    wide_tables
)

print("all_psf_wide shape:", all_psf_wide.shape)
print("unique key rows:", all_psf_wide[join_keys].drop_duplicates().shape[0])


In [ ]:
job_use = job_df[["empid", "job"]].drop_duplicates("empid")

model_df = all_psf_wide.merge(
    job_use,
    on="empid",
    how="left"
)

health_use = health_df[["empid", "age"]].drop_duplicates("empid")

model_df = model_df.merge(
    health_use,
    on="empid",
    how="left"
)

In [ ]:
target = "작업부하_answer"
target_quality_col = "작업부하_quality_100"
join_keys = ["empid", "survey_year", "survey_id", "set_id", "repeat_no"]

if target_quality_col in model_df.columns:
    df_model = model_df[
        model_df[target].notna()
        & (model_df[target_quality_col] == "성실")
    ].copy()
else:
    df_model = model_df[model_df[target].notna()].copy()

X = df_model.drop(
    columns=join_keys + [target],
    errors="ignore"
)

# 다른 PSF answer와 품질 라벨은 입력에서 제외
answer_cols = [c for c in X.columns if c.endswith("_answer")]
quality_cols = [
    c for c in X.columns
    if c.endswith(("_quality_100", "_quality_95", "_quality_90"))
]

X = X.drop(
    columns=answer_cols + quality_cols,
    errors="ignore"
)

y = df_model[target]
groups = df_model["empid"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("empid 수:", groups.nunique())
print("y 분포:")
print(y.value_counts())


In [ ]:
cat_features = ["job"]

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

cat_features = [
    c for c in cat_features
    if c in X.columns
]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

print("train/test rows:", len(train_idx), len(test_idx))
print("train/test empid:", len(train_groups), len(test_groups))
print("overlap empid:", len(train_groups & test_groups))

model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    verbose=100,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

pred = model.predict(X_test).ravel()

print(
    classification_report(
        y_test,
        pred,
        zero_division=0
    )
)


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_test,
    pred,
    labels=model.classes_
)

pd.DataFrame(
    cm,
    index=[f"실제_{c}" for c in model.classes_],
    columns=[f"예측_{c}" for c in model.classes_]
)

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

importance.head(20)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd

join_keys = ["empid", "survey_year", "survey_id", "set_id", "repeat_no"]
quality_suffixes = ("_quality_100", "_quality_95", "_quality_90")

results = {}
reports = {}

for target_psf in psf_tables.keys():
    target_col = f"{target_psf}_answer"
    target_quality_col = f"{target_psf}_quality_100"

    if target_col not in model_df.columns:
        print(f"skip: {target_psf}")
        continue

    df_model = model_df[model_df[target_col].notna()].copy()

    if target_quality_col in df_model.columns:
        df_model = df_model[
            df_model[target_quality_col] == "성실"
        ].copy()

    X = df_model.drop(
        columns=join_keys + [target_col],
        errors="ignore"
    )

    # 다른 PSF의 answer 컬럼과 품질 라벨은 입력에서 제외
    answer_cols = [c for c in X.columns if c.endswith("_answer")]
    quality_cols = [c for c in X.columns if c.endswith(quality_suffixes)]
    X = X.drop(
        columns=answer_cols + quality_cols,
        errors="ignore"
    )

    y = df_model[target_col]
    groups = df_model["empid"]

    if y.nunique() < 2:
        print(f"skip: {target_psf} - class가 1개뿐입니다.")
        continue

    cat_features = [
        c for c in ["job"]
        if c in X.columns
    ]

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=groups)
    )

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        verbose=False,
        random_seed=42
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    pred = model.predict(X_test).ravel()

    acc = accuracy_score(y_test, pred)
    macro_f1 = f1_score(y_test, pred, average="macro")
    weighted_f1 = f1_score(y_test, pred, average="weighted")

    importance = pd.DataFrame({
        "feature": X.columns,
        "importance": model.get_feature_importance()
    }).sort_values("importance", ascending=False)

    results[target_psf] = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "support": len(y_test),
        "train_empid": len(train_groups),
        "test_empid": len(test_groups),
        "overlap_empid": len(train_groups & test_groups),
        "model": model,
        "importance": importance
    }

    reports[target_psf] = classification_report(
        y_test,
        pred,
        output_dict=True,
        zero_division=0
    )

    print("=" * 60)
    print(target_psf)
    print("train/test rows:", len(train_idx), len(test_idx))
    print("train/test empid:", len(train_groups), len(test_groups))
    print("overlap empid:", len(train_groups & test_groups))
    print(classification_report(y_test, pred, zero_division=0))
    print(importance.head(20))


In [ ]:
df = psf_tables["작업부하"].copy()

if "quality_100" in df.columns:
    df = df[df["quality_100"] == "성실"].copy()

print("shape:", df.shape)
print("empid 수:", df["empid"].nunique())
print("survey_id 수:", df["survey_id"].nunique())

key_cols = [
    c for c in ["empid", "survey_id", "set_id", "repeat_no"]
    if c in df.columns
]

print("key 중복 수:", df.duplicated(key_cols).sum())
print(df.groupby(["empid", "survey_id"]).size().value_counts().sort_index())

single_drop_cols = [
    "empid",
    "survey_id",
    "survey_year",
    "set_id",
    "repeat_no",
    "PSF_name",
    "quality_100",
    "quality_95",
    "quality_90",
    "answer"
]

X = df.drop(
    columns=single_drop_cols,
    errors="ignore"
)

y = df["answer"]
groups = df["empid"]


In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("empid 수:", groups.nunique())
print("y 분포:")
print(y.value_counts())


# X2

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

print("train/test rows:", len(train_idx), len(test_idx))
print("train/test empid:", len(train_groups), len(test_groups))
print("overlap empid:", len(train_groups & test_groups))


In [ ]:
cat_features = []


In [ ]:
model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    verbose=100,
    random_seed=42
)


In [ ]:
model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)


In [ ]:
pred = model.predict(X_test).ravel()

print(
    classification_report(
        y_test,
        pred,
        zero_division=0
    )
)


In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
})

importance.sort_values(
    "importance",
    ascending=False
)

In [ ]:
# row가 예측, col이 실제 테이블
cm = confusion_matrix(
    y_test,
    pred,
    labels=model.classes_
)

pd.DataFrame(
    cm,
    index=model.classes_,
    columns=model.classes_
)

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
}).sort_values(
    "importance",
    ascending=False
)

importance

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# 1. 순서형 매핑
answer_order = {
    "매우나쁨": 0,
    "나쁨": 1,
    "보통": 2
}

# 2. pred가 2차원 배열이면 1차원으로 변환
pred_series = pd.Series(pred.ravel(), index=y_test.index)

# 3. 숫자로 변환
y_true_num = y_test.map(answer_order)
y_pred_num = pred_series.map(answer_order)

# 4. 결측 제거
valid_idx = y_true_num.notna() & y_pred_num.notna()

y_true_num = y_true_num[valid_idx]
y_pred_num = y_pred_num[valid_idx]

y_true_label = y_test[valid_idx]
y_pred_label = pred_series[valid_idx]

# 5. 일반 Accuracy
acc = accuracy_score(y_true_label, y_pred_label)

# 6. ±1 단계 허용 Accuracy
adjacent_acc = (np.abs(y_true_num - y_pred_num) <= 1).mean()

# 7. 완전 반대 방향 오류율
# 매우나쁨 <-> 보통처럼 2단계 이상 틀린 경우
severe_error_rate = (np.abs(y_true_num - y_pred_num) >= 2).mean()

# 8. Weighted Kappa
linear_kappa = cohen_kappa_score(
    y_true_num,
    y_pred_num,
    weights="linear"
)

quadratic_kappa = cohen_kappa_score(
    y_true_num,
    y_pred_num,
    weights="quadratic"
)

# 9. MAE / RMSE
mae = mean_absolute_error(y_true_num, y_pred_num)
rmse = np.sqrt(mean_squared_error(y_true_num, y_pred_num))

# 10. 결과 요약
metric_result = pd.DataFrame({
    "metric": [
        "Accuracy",
        "Adjacent Accuracy (±1)",
        "Severe Error Rate (>=2)",
        "Linear Weighted Kappa",
        "Quadratic Weighted Kappa",
        "Ordinal MAE",
        "Ordinal RMSE"
    ],
    "value": [
        acc,
        adjacent_acc,
        severe_error_rate,
        linear_kappa,
        quadratic_kappa,
        mae,
        rmse
    ]
})

metric_result

In [ ]:
from sklearn.metrics import confusion_matrix

labels_order = ["매우나쁨", "나쁨", "보통"]

cm = confusion_matrix(
    y_true_label,
    y_pred_label,
    labels=labels_order
)

cm_df = pd.DataFrame(
    cm,
    index=[f"실제_{x}" for x in labels_order],
    columns=[f"예측_{x}" for x in labels_order]
)

cm_df

In [ ]:
import numpy as np
import pandas as pd

proba = model.predict_proba(X_test)
classes = model.classes_

top2_idx = np.argsort(proba, axis=1)[:, -2:]
top2_labels = classes[top2_idx]

top2_acc = np.mean([
    y_true in top2
    for y_true, top2 in zip(y_test.values, top2_labels)
])

top2_acc

In [ ]:
risk_answers = ["나쁨", "매우나쁨"]

y_true_risk = y_test.isin(risk_answers)
y_pred_risk = pred_series.isin(risk_answers)

risk_acc = accuracy_score(y_true_risk, y_pred_risk)
risk_acc

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true_risk,
    y_pred_risk,
    target_names=["비위험", "위험"]
))